# Knuckledragger reals in the browser

Z3 and knuckledragger run entirely in this tab via Pyodide. The first cell installs the bundled wheels.

In [ ]:
%pip install -q --no-deps knuckledragger z3-solver lark graphviz

In [ ]:
import time

t = time.perf_counter()

In [ ]:
import kdrag as kd
import kdrag.smt as smt
import kdrag.theories.real as real
from kdrag.theories.real import rseq

print(f"imported in {time.perf_counter() - t:.1f}s")

In [ ]:
x, y = smt.Reals("x y")


@kd.Theorem(smt.ForAll([x, y], real.abs(x + y) <= real.abs(x) + real.abs(y)))
def abs_add_le(l):
    """Triangle inequality."""
    _x, _y = l.fixes()
    l.unfold(real.abs)
    l.auto()


abs_add_le

Limits of sequences, using lemmas from `kdrag.theories.real.rseq`.

In [ ]:
a = smt.Const("a", real.RSeq)
L, c = smt.Reals("L c")


@kd.Theorem(smt.ForAll([a, L, c], rseq.has_lim(a, L), rseq.has_lim(a + rseq.const(c), L + c)))
def has_lim_add_const(l):
    """Shifting a convergent sequence shifts its limit."""
    _a, _L, _c = l.fixes()
    l.intro()
    l.auto(by=[rseq.has_lim_add, rseq.has_lim_const])


has_lim_add_const

A false claim gets a countermodel.

In [ ]:
try:
    kd.prove(smt.ForAll([x], x * x > 0))
except kd.kernel.LemmaError as e:
    print(type(e).__name__, e)